In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install transformers datasets rouge-score sentencepiece -q
print("Done!")

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

from rouge_score import rouge_scorer
from tqdm import tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GPU:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

# **Finetuning Bart for headline generation**


In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kartik1321/inshort-news-summary/clean_data.csv")
df = df[['headlines','text']].dropna()
df = df[df['text'].str.split().str.len() >= 30]
df = df[df['headlines'].str.split().str.len() >= 5]
df = df.reset_index(drop = True)
print('shape: ',df.shape)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Val:  ", len(val_df))
print("Test: ", len(test_df))

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")
print("Tokenizer loaded!")

In [ ]:
class InshortsSumDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=512, max_target=60):
        self.df        = dataframe
        self.tokenizer = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        article  = str(self.df["text"][idx])
        headline = str(self.df["headlines"][idx])

        inputs = self.tokenizer(
            article,
            max_length=self.max_input,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            headline,
            max_length=self.max_target,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels":         labels
        }


In [ ]:
# create datasets
train_dataset = InshortsSumDataset(train_df, tokenizer)
val_dataset   = InshortsSumDataset(val_df,   tokenizer)
test_dataset  = InshortsSumDataset(test_df,  tokenizer)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False, num_workers=2)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))

In [ ]:
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded on: {device}")
print(f"Trainable parameters: {total_params:,}")

In [ ]:
optimizer    = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps  = len(train_loader) * 3
warmup_steps = 500

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Optimizer ready!")
print(f"Total training steps: {total_steps:,}")

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for i, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if i % 500 == 0:
            print(f"  Step {i}/{len(loader)} — loss: {loss.item():.4f}")

    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
    return total_loss / len(loader)

print("Functions ready!")

In [ ]:
EPOCHS = 3
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 30)

    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss   = validate(model, val_loader, device)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    # save best model after every epoch
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained("/kaggle/working/bart_inshorts")
        tokenizer.save_pretrained("/kaggle/working/bart_inshorts")
        print("✓ Best model saved!")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/bart_model", "zip", "/kaggle/working", "bart_inshorts")
print("Done!")

In [ ]:
model_path = "/kaggle/input/datasets/kartik1321/bart-inshorts-model/bart_inshorts"

tokenizer = BartTokenizer.from_pretrained(model_path)
model     = BartForConditionalGeneration.from_pretrained(model_path)
model     = model.to(device)
model.eval()

print("Model loaded on:", device)

In [ ]:
def generate_summary(text):
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=4,           # beam search
        max_length=60,
        min_length=8,
        length_penalty=2.0,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Model ready for inference!")

In [ ]:
for i in range(5):
    article  = test_df["text"][i]
    actual   = test_df["headlines"][i]
    generated = generate_summary(article)

    print(f"Article {i+1}:")
    print(f"  Actual:    {actual}")
    print(f"  Generated: {generated}")
    print()

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []

for i in tqdm(range(500)):
    try:
        generated = generate_summary(test_df["text"][i])
        actual    = test_df["headlines"][i]
        scores    = scorer.score(actual, generated)
        results.append({
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure
        })
    except:
        pass

avg_rouge1 = sum(r["rouge1"] for r in results) / len(results)
avg_rouge2 = sum(r["rouge2"] for r in results) / len(results)
avg_rougeL = sum(r["rougeL"] for r in results) / len(results)

print(f"BART ROUGE-1: {avg_rouge1:.3f}")
print(f"BART ROUGE-2: {avg_rouge2:.3f}")
print(f"BART ROUGE-L: {avg_rougeL:.3f}")

In [ ]:
import json

results_table = {
    "textrank": {"rouge1": 0.095, "rouge2": 0.023, "rougeL": 0.074},
    "bart":     {"rouge1": 0.577, "rouge2": 0.343, "rougeL": 0.534}
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results_table, f, indent=4)

print("Results saved!")

# **finetuning bart on different dataset(news summary generation)** 


In [ ]:
df2 = pd.read_csv("/kaggle/input/datasets/kartik1321/inshort-dataset/clean_data_v2.csv")
df2 = df2.dropna()
df2 = df2.reset_index(drop=True)

print("Shape:", df2.shape)
print("\nSample article:", df2["article"][0][:150])
print("\nSample summary:", df2["summary"][0])

In [ ]:
train_df2, temp_df2 = train_test_split(df2, test_size=0.20, random_state=42)
val_df2, test_df2   = train_test_split(temp_df2, test_size=0.50, random_state=42)

train_df2 = train_df2.reset_index(drop=True)
val_df2   = val_df2.reset_index(drop=True)
test_df2  = test_df2.reset_index(drop=True)

print("Train:", len(train_df2))
print("Val:  ", len(val_df2))
print("Test: ", len(test_df2))

In [ ]:
tokenizer2 = BartTokenizer.from_pretrained("facebook/bart-base")
print("Tokenizer loaded! Device:", device)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class InshortsSumDataset2(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=1024, max_target=128):
        self.df         = dataframe
        self.tokenizer  = tokenizer
        self.max_input  = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        article = str(self.df["article"][idx])
        summary = str(self.df["summary"][idx])

        inputs = self.tokenizer(
            article,
            max_length=self.max_input,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets = self.tokenizer(
            summary,
            max_length=self.max_target,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels":         labels
        }

train_dataset2 = InshortsSumDataset2(train_df2, tokenizer2)
val_dataset2  = InshortsSumDataset2(val_df2,   tokenizer2)
test_dataset2  = InshortsSumDataset2(test_df2,  tokenizer2)

print(f"Train: {len(train_dataset2)} | Val: {len(val_dataset2)} | Test: {len(test_dataset2)}")

In [ ]:
train_loader2 = DataLoader2(train_dataset2, batch_size=4, shuffle=True,  num_workers=2)
val_loader2   = DataLoader2(val_dataset2,   batch_size=4, shuffle=False, num_workers=2)
test_loader2  = DataLoader2(test_dataset2,  batch_size=4, shuffle=False, num_workers=2)

print("Train batches:", len(train_loader2))
print("Val batches:  ", len(val_loader2))

In [ ]:
model2 = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
model2 = model2.to(device)

total_params2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Model loaded on: {device}")
print(f"Trainable parameters: {total_params2:,}")

In [ ]:
optimizer2    = AdamW(model2.parameters(), lr=2e-5, weight_decay=0.01)
total_steps2  = len(train_loader2) * 5   # 5 epochs this time (small dataset)
warmup_steps2 = 100

scheduler2 = get_linear_schedule_with_warmup(
    optimizer2,
    num_warmup_steps=warmup_steps2,
    num_training_steps=total_steps2
)

print("Optimizer ready!")
print(f"Total training steps: {total_steps2:,}")

In [ ]:
def train_epoch2(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for i, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

        if i % 100 == 0:
            print(f"  Step {i}/{len(loader)} — loss: {loss.item():.4f}")

    return total_loss / len(loader)


def validate2(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
    return total_loss / len(loader)

print("Functions ready!")

In [ ]:
EPOCHS        = 5
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("-" * 30)

    train_loss = train_epoch2(model2, train_loader2, optimizer2, scheduler2, device)
    val_loss   = validate2(model2, val_loader2, device)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained("/kaggle/working/bart_inshorts_v2")
        tokenizer.save_pretrained("/kaggle/working/bart_inshorts_v2")
        print("✓ Best model saved!")

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/bart_inshorts_v2", "zip", "/kaggle/working/bart_inshorts_v2")
print("Zip ready!")

In [ ]:
model_path2 = "/kaggle/input/datasets/kartik1321/inshort-model-v2"

tokenizer2 = BartTokenizer.from_pretrained(model_path2)
model2     = BartForConditionalGeneration.from_pretrained(model_path2)
model2     = model2.to(device)
model2.eval()

print("Model loaded on:", device)

In [ ]:
def generate_summary2(text):
    inputs = tokenizer2(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model2.generate(
        inputs["input_ids"],
        num_beams=4,
        max_length=180,      
        min_length=50,
        length_penalty=3.0,   
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    return tokenizer2.decode(summary_ids[0], skip_special_tokens=True)

print("Model ready for inference!")

In [ ]:
# test on 3 samples
for i in range(3):
    article   = test_df2["article"][i]
    actual    = test_df2["summary"][i]
    generated = generate_summary2(article)

    print(f"Sample {i+1}")
    print(f"Actual:    {actual}")
    print(f"Generated: {generated}")
    print(f"Words:     {len(generated.split())}")
    print()

In [ ]:
scorer2 = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results2 = []

for i in tqdm(range(500)):
    try:
        generated = generate_summary2(test_df2["article"][i])
        actual    = test_df2["summary"][i]
        scores    = scorer2.score(actual, generated)
        results2.append({
            "rouge1": scores["rouge1"].fmeasure,
            "rouge2": scores["rouge2"].fmeasure,
            "rougeL": scores["rougeL"].fmeasure
        })
    except:
        pass

avg_rouge1 = sum(r["rouge1"] for r in results) / len(results)
avg_rouge2 = sum(r["rouge2"] for r in results) / len(results)
avg_rougeL = sum(r["rougeL"] for r in results) / len(results)

print(f"BART ROUGE-1: {avg_rouge1:.3f}")
print(f"BART ROUGE-2: {avg_rouge2:.3f}")
print(f"BART ROUGE-L: {avg_rougeL:.3f}")

In [ ]:
import json

results_table = {
    "textrank": {"rouge1": 0.095, "rouge2": 0.023, "rougeL": 0.074},
    "bart_headline": {"rouge1": 0.577, "rouge2": 0.343, "rougeL": 0.534},
    "bart_summary":  {"rouge1": 0.482, "rouge2": 0.267, "rougeL": 0.363}
}

with open("/kaggle/working/results.json", "w") as f:
    json.dump(results_table, f, indent=4)

print("Results saved!")

# using a bigger pretrained model: Facebook/bart-large-cnn

In [ ]:
# load facebook's pretrained summarisation model
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model     = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
bart_model     = bart_model.to(device)
bart_model.eval()

print("bart-large-cnn loaded!")

In [ ]:
def generate_summary_large(text):
    inputs = bart_tokenizer(
        text,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = bart_model.generate(
        inputs["input_ids"],
        num_beams=5,
        max_length=130,
        min_length=70,
        length_penalty=3.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# test on ISI article
isi_article = df2['article'][24]
summary = generate_summary_large(isi_article)
print("Summary:", summary)
print("Words:", len(summary.split()))

In [ ]:
!pip install sumy


# **Comparing Textrank, BART v2, Bart-large-cnn**

In [ ]:

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

article = df2['article'][24]
# Model 1 - TextRank
def textrank_summarize(text, num_sentences=2):
    parser    = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = TextRankSummarizer()
    summary   = summarizer(parser.document, num_sentences)
    return " ".join([str(sentence) for sentence in summary])

# compare all 3
print("=" * 60)
print("ORIGINAL ARTICLE:")
print(article[:200], "...")
print("=" * 60)

print("\n1️⃣  TextRank (extractive):")
print(textrank_summarize(article))

print("\n2️⃣  BART v2 (your fine-tuned model):")
print(generate_summary2(article))

print("\n3️⃣  bart-large-cnn (Facebook pretrained):")
print(generate_summary_large(article))
print("=" * 60)